# AiXbio — Notebook 3: SAE Feature Discovery

Uses interPLM pre-trained SAEs on ESM-2-650M to find mechanistic toxin-detection features.
Experiment 4: which SAE features fire selectively on toxins, and do they generalise to redesigns?

In [ ]:
import warnings, json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from Bio import SeqIO
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BEST_LAYER = 24    # interPLM-esm2-650m has SAEs at layers 4,8,12,16,20,24,28,30,33
                   # Override if your Notebook 2 found a different best layer
Path('results').mkdir(exist_ok=True)
print(f'Device: {DEVICE}  |  SAE layer: {BEST_LAYER}')

## 1. Load interPLM SAE (ESM-2-650M, layer 24)

In [ ]:
from interplm.sae.inference import load_sae_from_hf

# Load SAE for ESM-2-650M at layer 24 (matches our embedding layer)
sae = load_sae_from_hf(plm_model='esm2-650m', plm_layer=BEST_LAYER)
sae = sae.to(DEVICE).eval()

# Inspect architecture
print(f'SAE type: {type(sae).__name__}')
print(f'Parameters: {sum(p.numel() for p in sae.parameters()):,}')

# Get input/feature dimensions from weight shapes
for name, param in sae.named_parameters():
    print(f'  {name}: {tuple(param.shape)}')

## 2. Get SAE Feature Activations from Pre-computed Embeddings

In [ ]:
def get_sae_features(embeddings: np.ndarray, batch_size: int = 256) -> np.ndarray:
    """
    Run pre-computed ESM-2 embeddings through the SAE encoder.
    Returns feature activation matrix (n_seq, n_features).
    """
    all_acts = []
    with torch.no_grad():
        for i in range(0, len(embeddings), batch_size):
            batch = torch.tensor(embeddings[i:i+batch_size],
                                  dtype=torch.float32).to(DEVICE)
            # interPLM SAE: encode() returns feature activations (post-ReLU)
            try:
                acts = sae.encode(batch)           # preferred method
            except AttributeError:
                acts = sae(batch)[1]               # fallback: (recon, acts, ...) tuple
            all_acts.append(acts.cpu().float().numpy())
    return np.concatenate(all_acts, axis=0)


# Load pre-computed embeddings (from Notebook 1)
tox_embs  = np.load(f'embeddings/natural_toxins_layer{BEST_LAYER}.npy')
ctrl_embs = np.load(f'embeddings/controls_layer{BEST_LAYER}.npy')
rdsg_embs = np.load(f'embeddings/redesigns_layer{BEST_LAYER}.npy')

print(f'Toxins:   {tox_embs.shape}  Controls: {ctrl_embs.shape}  Redesigns: {rdsg_embs.shape}')

print('Running SAE encoder on natural toxins...')
tox_acts  = get_sae_features(tox_embs)
print('Running SAE encoder on controls...')
ctrl_acts = get_sae_features(ctrl_embs)
print('Running SAE encoder on redesigns...')
rdsg_acts = get_sae_features(rdsg_embs)

n_features = tox_acts.shape[1]
print(f'\nFeature matrix shapes:')
print(f'  Toxins:   {tox_acts.shape}')
print(f'  Controls: {ctrl_acts.shape}')
print(f'  Redesigns:{rdsg_acts.shape}')
print(f'  Total features: {n_features:,}')

## 3. Identify Toxin-Discriminating Features

In [ ]:
# Train/test split on natural sequences
X_nat = np.concatenate([tox_acts, ctrl_acts])
y_nat = np.concatenate([np.ones(len(tox_acts)), np.zeros(len(ctrl_acts))])
idx = np.arange(len(X_nat))
tr_idx, te_idx = train_test_split(idx, test_size=0.2, stratify=y_nat, random_state=42)

X_tr, y_tr = X_nat[tr_idx], y_nat[tr_idx]
X_te, y_te = X_nat[te_idx], y_nat[te_idx]

# Per-feature AUROC on training set
print(f'Computing per-feature AUROC over {n_features:,} features...')
feat_aurocs = np.zeros(n_features)
feat_active = np.zeros(n_features)  # fraction of sequences where feature fires

for f in range(n_features):
    scores = X_tr[:, f]
    if scores.max() > 0 and len(np.unique(scores)) > 1:
        try:
            feat_aurocs[f] = roc_auc_score(y_tr, scores)
        except:
            feat_aurocs[f] = 0.5
    else:
        feat_aurocs[f] = 0.5
    feat_active[f] = (scores > 0).mean()

# Reflect AUROC below 0.5 (anti-toxin features are also informative)
feat_aurocs_abs = np.where(feat_aurocs >= 0.5, feat_aurocs, 1 - feat_aurocs)

# Top-K toxin features
TOP_K = 50
top_feat_idx = np.argsort(feat_aurocs_abs)[::-1][:TOP_K]

print(f'\nTop {TOP_K} toxin features:')
print(f'  AUROC range:    {feat_aurocs_abs[top_feat_idx[0]]:.4f} – {feat_aurocs_abs[top_feat_idx[-1]]:.4f}')
print(f'  Activation rate:{feat_active[top_feat_idx].mean()*100:.1f}% of training sequences')
print(f'  Dead features:  {(feat_aurocs == 0.5).sum():,}/{n_features:,}')

# Save feature stats
feat_stats = {
    'top_feature_indices': top_feat_idx.tolist(),
    'top_feature_aurocs':  feat_aurocs_abs[top_feat_idx].tolist(),
    'top_feature_activation_rates': feat_active[top_feat_idx].tolist(),
    'all_aurocs_mean': float(feat_aurocs_abs.mean()),
    'n_features': int(n_features),
    'n_active_features': int((feat_active > 0).sum()),
    'top_k': TOP_K,
}
with open('results/sae_feature_stats.json', 'w') as f:
    json.dump(feat_stats, f, indent=2)
print('Saved → results/sae_feature_stats.json')

## 4. SAE-Probe: Classify Using Top-K Features Only

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ── Full embedding probe (baseline) ──────────────────────────
sc_full = StandardScaler()
X_tr_full = sc_full.fit_transform(X_tr)
X_te_full = sc_full.transform(X_te)
lr_full = LogisticRegression(C=1.0, max_iter=500).fit(X_tr_full, y_tr)
auroc_full = roc_auc_score(y_te, lr_full.predict_proba(X_te_full)[:, 1])

# ── Top-K SAE features probe ──────────────────────────────────
sc_sae = StandardScaler()
X_tr_sae = sc_sae.fit_transform(X_tr[:, top_feat_idx])
X_te_sae = sc_sae.transform(X_te[:, top_feat_idx])
lr_sae = LogisticRegression(C=1.0, max_iter=500).fit(X_tr_sae, y_tr)
auroc_sae = roc_auc_score(y_te, lr_sae.predict_proba(X_te_sae)[:, 1])

print(f'AUROC comparison (natural test set):')
print(f'  Full embedding ({n_features:,} features): {auroc_full:.4f}')
print(f'  Top-{TOP_K} SAE features:              {auroc_sae:.4f}')
print(f'  Compression ratio: {n_features/TOP_K:.0f}× fewer features')

## 5. Generalisation Curve — SAE Features vs Probe vs BLAST

In [ ]:
seq_identities   = np.load('embeddings/sequence_identities.npy')
blast_identities = np.load('embeddings/blast_identities.npy')

# Build full test set: natural test + redesigns
# Redesigns are all toxin class (label=1)
X_test_nat  = X_te
y_test_nat  = y_te
X_test_rdsg = rdsg_acts[:, top_feat_idx]
y_test_rdsg = np.ones(len(rdsg_acts))

X_test_all_sae  = np.concatenate([X_test_nat, sc_sae.transform(rdsg_acts[:, top_feat_idx])])
X_test_all_full = np.concatenate([X_te_full, sc_full.transform(rdsg_acts)])
y_test_all      = np.concatenate([y_test_nat, y_test_rdsg])

IDENTITY_BINS = [
    (0.90,1.00,'90–100%'),(0.70,0.90,'70–90%'),(0.50,0.70,'50–70%'),
    (0.30,0.50,'30–50%'),(0.10,0.30,'10–30%'),(0.00,0.10,'<10%'),
]

def eval_bin_classifier(clf, X_all, y_all, mask):
    if mask.sum() < 5 or len(np.unique(y_all[mask])) < 2:
        return float('nan')
    return roc_auc_score(y_all[mask], clf.predict_proba(X_all[mask])[:, 1])

def blast_sens(mask, thr=0.30):
    if mask.sum() < 5: return float('nan')
    y = y_test_all[mask]; pred = (blast_identities[mask] >= thr).astype(int)
    tp = ((pred==1)&(y==1)).sum(); fn = ((pred==0)&(y==1)).sum()
    return float(tp/(tp+fn)) if (tp+fn)>0 else float('nan')

sae_curve = []
print(f'{"Bin":10s}  {"n":>5}  {"SAE probe":>10}  {"Full probe":>10}  {"BLAST":>8}')
print('-'*52)
for lo, hi, label in IDENTITY_BINS:
    mask = (seq_identities >= lo) & (seq_identities < hi)
    auroc_s = eval_bin_classifier(lr_sae,  X_test_all_sae,  y_test_all, mask)
    auroc_f = eval_bin_classifier(lr_full, X_test_all_full, y_test_all, mask)
    bl = blast_sens(mask)
    sae_curve.append({'bin': label, 'lo': lo, 'hi': hi,
                       'n': int(mask.sum()),
                       'sae_auroc': auroc_s, 'full_auroc': auroc_f,
                       'blast_sensitivity': float(bl)})
    print(f"{label:10s}  {int(mask.sum()):>5}  {auroc_s:>10.3f}  {auroc_f:>10.3f}  {bl:>8.3f}")

print(f'\nKey finding: SAE probe maintains AUROC > 0.7 down to <30% identity? '
      f'{"YES" if any(e["sae_auroc"] > 0.7 for e in sae_curve if e["lo"] < 0.3) else "NO"}')

## 6. Feature Robustness Analysis

In [ ]:
# Per-feature: does the feature transfer to redesigns?
# Compare activation rate on toxins vs redesigns
print('Feature transfer analysis (top 20 features):')
print(f'{"Feat":>6}  {"Train AUROC":>11}  {"Tox act%":>9}  {"Ctrl act%":>9}  {"Rdsg act%":>9}  {"Transfer":>9}')
print('-'*62)

transfer_results = []
for rank, f in enumerate(top_feat_idx[:20]):
    tox_rate  = (tox_acts[:, f]  > 0).mean()
    ctrl_rate = (ctrl_acts[:, f] > 0).mean()
    rdsg_rate = (rdsg_acts[:, f] > 0).mean()
    # Transfer: does redesign activation ≈ toxin activation?
    transfer = rdsg_rate / (tox_rate + 1e-6)
    auroc_f = feat_aurocs_abs[f]
    print(f"{f:>6}  {auroc_f:>11.4f}  {tox_rate*100:>8.1f}%  {ctrl_rate*100:>8.1f}%  {rdsg_rate*100:>8.1f}%  {transfer:>9.2f}")
    transfer_results.append({'feature': int(f), 'rank': rank+1,
                              'train_auroc': float(auroc_f),
                              'tox_activation': float(tox_rate),
                              'ctrl_activation': float(ctrl_rate),
                              'redesign_activation': float(rdsg_rate),
                              'transfer_ratio': float(transfer)})

mean_transfer = np.mean([r['transfer_ratio'] for r in transfer_results])
print(f'\nMean transfer ratio: {mean_transfer:.2f}')
print('  > 0.8 = feature generalises to redesigns (structurally robust)')
print('  < 0.3 = feature is sequence-identity dependent (evadable)')

## 7. Save All Results

In [ ]:
# Load main_results if it exists (from Notebook 2)
try:
    with open('results/main_results.json') as f:
        main_results = json.load(f)
    has_main = True
except FileNotFoundError:
    main_results = {}; has_main = False

sae_results = {
    'sae_model': f'interPLM-esm2-650m-layer{BEST_LAYER}',
    'n_features_total': int(n_features),
    'n_active_features': int((feat_active > 0).sum()),
    'top_k': TOP_K,
    'auroc_full_embedding': float(auroc_full),
    'auroc_sae_top_k': float(auroc_sae),
    'compression_ratio': float(n_features / TOP_K),
    'mean_transfer_ratio': float(mean_transfer),
    'feature_stats': feat_stats,
    'sae_generalisation_curve': sae_curve,
    'feature_transfer': transfer_results,
}

def _conv(o):
    if isinstance(o, (np.integer, np.floating)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return o

with open('results/sae_results.json', 'w') as f:
    json.dump(sae_results, f, default=_conv, indent=2)

# Merge into main_results for Notebook 4
main_results['sae'] = sae_results
with open('results/main_results.json', 'w') as f:
    json.dump(main_results, f, default=_conv, indent=2)

print('Saved → results/sae_results.json')
print('Merged → results/main_results.json')
print()
print('=== SAE Summary ===')
print(f'  Active features: {sae_results["n_active_features"]:,}/{n_features:,}')
print(f'  Full AUROC:      {auroc_full:.4f}')
print(f'  SAE-top-{TOP_K}:    {auroc_sae:.4f}  ({n_features//TOP_K}× compression)')
print(f'  Transfer ratio:  {mean_transfer:.2f} (1.0 = perfect generalisation)')
print(f'  → Ready for Notebook 4 figures')